# Day 10: GRPO 实战 -- 复现 DeepSeek-R1 训练思路

**目标：** 深入理解 GRPO 算法，用 GSM8K 数学数据集训练能"思考"的模型。

**学习路线：**
1. DeepSeek-R1 的意义与突破
2. GRPO 算法详解
3. Reward 函数设计
4. GSM8K 实战训练
5. 涌现现象分析

> **前置知识：** Day 09 RL 基础

---
## 1. DeepSeek-R1 的意义

### 1.1 为什么重要？

DeepSeek-R1 证明：**纯 RL 训练可以让模型涌现推理能力**。

传统路线：`预训练 -> SFT (人写CoT) -> RLHF`

DeepSeek-R1：`预训练 -> GRPO (纯RL，无SFT)`

关键发现：
- 模型**自发学会**使用 `<think>...</think>` 标签
- 无需人工标注思维链数据
- 推理能力随训练逐渐涌现

### 1.2 涌现现象

| 训练阶段 | 模型行为 | 说明 |
|---------|---------|------|
| 初期 | 直接输出答案 | 还没学会思考 |
| 中期 | 开始出现简单推理 | 发现思考有助于高奖励 |
| 后期 | 自发使用 think 标签 | 结构化思考涌现 |
| 成熟期 | 多步骤、有验算 | 完整思维链 |

In [ ]:
# 环境准备
import sys, re, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "day10_grpo" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(42)
print("环境准备完成!")

In [ ]:
# DeepSeek-R1 前后对比
print("=" * 60)
print("DeepSeek-R1 训练前后对比")
print("=" * 60)

question = "小明有15个苹果，又买了8个，一共有多少个苹果？"
print(f"\n问题: {question}")

print("\n--- 训练前 ---")
print("回复: 小明有15个苹果加上8个等于23个苹果")
print("特点: 直接给答案，无推理，格式随意")

print("\n--- 训练后 (GRPO) ---")
print("<think>")
print("已知: 小明有 15 个苹果")
print("新增: 又买了 8 个")
print("计算: 15 + 8 = 23")
print("验算: 23 - 8 = 15 ok")
print("</think>")
print("答案: 23")
print("特点: 自发使用<think>，有推理和验算")

---
## 2. GRPO 算法详解

### 2.1 完整流程

```
对每个 prompt x:
  1. 采样 G 个回复
  2. 计算奖励 r_j = R(y_j, x)
  3. 组内归一化: A_j = (r_j - mean) / std
  4. 策略梯度更新（带 clipping + KL 惩罚）
```

### 2.2 关键公式

GRPO 目标 = (1/G) * sum min(rho * A, clip(rho) * A) - beta * KL

- rho = pi_new / pi_old（概率比）
- A_hat = (r_i - mean) / std（组内归一化优势，核心创新）
- clip：限制更新幅度
- KL：防止偏离参考模型

In [ ]:
# GRPO 组内归一化详解
print("=" * 60)
print("GRPO 组内归一化详解")
print("=" * 60)

G = 8
np.random.seed(42)
responses = [
    ("直接答案", 0.15), ("<think>简单</think>答案:对", 0.72),
    ("乱回复", 0.05), ("<think>\n步骤1\n步骤2\n</think>\n答案:对", 0.95),
    ("有点推理", 0.40), ("<think>推理</think>答案:错", 0.30),
    ("<think>\n详细\n验算\n</think>\n答案:对", 0.88), ("部分正确", 0.25),
]

rewards = np.array([r for _, r in responses])
mean_r = np.mean(rewards); std_r = np.std(rewards)
advantages = (rewards - mean_r) / (std_r + 1e-8)

print(f"\n组内统计: 均值={mean_r:.3f}, 标准差={std_r:.3f}")
print(f"\n{'编号':>4} | {'奖励':>6} | {'优势':>8} | {'操作':>10} | 回复")
print("-" * 65)
for i, ((desc, r), adv) in enumerate(zip(responses, advantages)):
    act = "提高概率" if adv > 0 else "降低概率"
    print(f"  {i:2d} | {r:6.3f} | {adv:+8.3f} | {act:>10} | {desc[:20]}")

In [ ]:
# 组内归一化可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 原始奖励
ax = axes[0]
colors_r = plt.cm.RdYlGn(rewards)
ax.bar(range(G), rewards, color=colors_r, alpha=0.8)
ax.axhline(y=mean_r, color="red", linestyle="--", label=f"均值={mean_r:.2f}")
ax.set_title("Step 1: 原始奖励"); ax.set_xlabel("回复编号"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3)

# 归一化优势
ax = axes[1]
colors_a = ["#4ECDC4" if a > 0 else "#FF6B6B" for a in advantages]
ax.bar(range(G), advantages, color=colors_a, alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Step 2: 归一化优势"); ax.set_xlabel("回复编号"); ax.set_ylabel("优势"); ax.grid(True, alpha=0.3)

# 概率更新
ax = axes[2]
init_p = 1.0 / G; lr_s = 0.05
new_p = init_p + lr_s * advantages; new_p = np.clip(new_p, 0.01, 1); new_p /= new_p.sum()
x = np.arange(G); w = 0.35
ax.bar(x-w/2, [init_p]*G, w, label="更新前", color="#FFD93D", alpha=0.8)
ax.bar(x+w/2, new_p, w, label="更新后", color="#4ECDC4", alpha=0.8)
ax.set_title("Step 3: 概率更新"); ax.set_xlabel("回复编号"); ax.set_ylabel("概率"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("GRPO 三步流程", fontsize=14, fontweight="bold"); plt.tight_layout(); plt.show()

### 2.3 Reward 函数设计

| 奖励类型 | 评判内容 | 权重 | 作用 |
|---------|---------|------|------|
| **格式奖励** | 是否用 think 标签、多步推理 | 0.3 | 引导结构化思考 |
| **正确性奖励** | 最终答案是否正确 | 0.7 | 确保结果正确 |

In [ ]:
# 格式奖励函数
def format_reward(response):
    score = 0.0
    has_open = "<think>" in response
    has_close = "</think>" in response
    if has_open and has_close:
        score += 0.3
        content = re.findall(r"<think>(.*?)</think>", response, re.DOTALL)
        if content and content[0].count("\n") >= 2:
            score += 0.2
    elif has_open or has_close:
        score += 0.1
    if any(k in response for k in ["答案", "结果", "="]):
        score += 0.2
    if has_open and has_close and any(k in response for k in ["答案", "结果", "="]):
        if response.find("</think>") < len(response) - 5:
            score += 0.3
    return min(score, 1.0)

# 正确性奖励函数
def correctness_reward(response, expected):
    nums = re.findall(r"[-+]?\d*\.?\d+", response)
    exp = expected.strip().replace(",", "")
    if not nums: return 0.0
    if nums[-1] == exp: return 1.0
    if exp in nums: return 0.5
    return 0.0

def combined_reward(response, expected, fw=0.3, cw=0.7):
    return fw * format_reward(response) + cw * correctness_reward(response, expected)

print("奖励函数定义完成")

In [ ]:
# 奖励函数演示
print("=" * 60)
print("奖励函数效果演示")
print("=" * 60)

cases = [
    ("无格式直答", "23", "23"),
    ("部分格式", "<think>15+8</think> 23", "23"),
    ("完整格式+正确", "<think>\n15+8=23\n验算正确\n</think>\n答案: 23", "23"),
    ("完整格式+错误", "<think>\n15+8=22\n</think>\n答案: 22", "23"),
    ("无格式+错误", "我不知道", "23"),
]

print(f"\n{'场景':^16} | {'格式':>6} | {'正确性':>8} | {'综合':>6}")
print("-" * 50)
for name, resp, ans in cases:
    f = format_reward(resp); c = correctness_reward(resp, ans); t = combined_reward(resp, ans)
    print(f"{name:^16} | {f:>6.2f} | {c:>8.2f} | {t:>6.2f}")

---
## 3. GSM8K 实战

GSM8K 是 OpenAI 发布的小学数学应用题数据集。

### 3.1 数据集探索

In [ ]:
# 准备数据
def create_synthetic_math_data(num=100):
    np.random.seed(42)
    data = []
    templates = [
        ("小明有 {a} 个苹果，又买了 {b} 个，一共有多少个？", lambda a, b: a + b),
        ("{a} 个学生每人分 {b} 支笔，共需多少支？", lambda a, b: a * b),
        ("绳子长 {a} 米剪去 {b} 米还剩多少？", lambda a, b: a - b),
        ("{a} 个糖果分给 {b} 人每人多少个？", lambda a, b: a // b if b > 0 else 0),
    ]
    for i in range(num):
        t, fn = templates[i % len(templates)]
        a = np.random.randint(10, 100); b = np.random.randint(1, min(a, 20))
        data.append({"question": t.format(a=a, b=b), "expected_answer": str(fn(a, b))})
    return data

try:
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main", split="train[:50]")
    dataset = []
    for ex in ds:
        ans = ex["answer"].split("####")[-1].strip().replace(",", "") if "####" in ex["answer"] else "0"
        dataset.append({"question": ex["question"], "expected_answer": ans})
    print(f"GSM8K: {len(dataset)} 条")
except:
    dataset = create_synthetic_math_data(100)
    print(f"合成数据: {len(dataset)} 条")

In [ ]:
# 数据集探索
print("数据集样例")
for i, item in enumerate(dataset[:5]):
    print(f"\n[样例 {i+1}]")
    print(f"  问题: {item['question']}")
    print(f"  答案: {item['expected_answer']}")

q_lens = [len(d["question"]) for d in dataset]
print(f"\n统计: 总量={len(dataset)}, 平均长度={np.mean(q_lens):.1f}")

### 3.2 GRPOTrainer 配置与训练

In [ ]:
# 检查训练依赖
import torch
can_train = False
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOTrainer, GRPOConfig
    can_train = True
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"依赖通过! 设备: {device}")
except ImportError as e:
    print(f"缺少依赖: {e}"); print("使用模拟模式")
    device = "cpu"

In [ ]:
# 真实训练或模拟
if can_train and device == "cuda":
    print("真实 GRPO 训练模式")
    model_name = "Qwen/Qwen2-0.5B"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, trust_remote_code=True, device_map="auto")
    print(f"模型加载完成: {model_name}")
    print("(完整训练代码见 grpo_training.py)")
else:
    print("模拟模式 (无GPU或缺少依赖)")

In [ ]:
# GRPO 模拟训练
print("\nGRPO 训练模拟")
G, num_prompts, num_epochs = 4, 50, 8
quality = 0.3
all_mean, all_fmt, all_cor = [], [], []

for epoch in range(num_epochs):
    erews, efmt, ecor = [], [], []
    for p in range(num_prompts):
        for g in range(G):
            fr = np.random.beta(quality*5+1, 3); cr = np.random.beta(quality*3+1, 4)
            erews.append(0.3*fr + 0.7*cr); efmt.append(fr); ecor.append(cr)
    all_mean.append(np.mean(erews)); all_fmt.append(np.mean(efmt)); all_cor.append(np.mean(ecor))
    quality = min(quality + 0.1, 0.95)
    print(f"  Epoch {epoch+1}/{num_epochs} | 综合: {all_mean[-1]:.4f} | 格式: {all_fmt[-1]:.4f} | 正确性: {all_cor[-1]:.4f}")

In [ ]:
# 训练曲线可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ep = range(1, num_epochs+1)
ax.plot(ep, all_mean, "b-o", label="综合", ms=5); ax.plot(ep, all_fmt, "g-s", label="格式", ms=5)
ax.plot(ep, all_cor, "r-^", label="正确性", ms=5)
ax.set_title("GRPO 训练奖励"); ax.set_xlabel("Epoch"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(np.random.beta(2,5,200), bins=25, alpha=0.6, label="训练前", color="#FF6B6B")
ax.hist(np.random.beta(5,2,200), bins=25, alpha=0.6, label="训练后", color="#4ECDC4")
ax.set_title("格式奖励分布变化"); ax.set_xlabel("格式奖励"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
np.random.seed(123); gr = np.random.normal(0.5, 0.2, 8)
advs = (gr - np.mean(gr)) / np.std(gr)
ax.bar(range(len(advs)), advs, color=["#4ECDC4" if a>0 else "#FF6B6B" for a in advs], alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("组内归一化优势"); ax.set_xlabel("回复编号"); ax.grid(True, alpha=0.3, axis="y")

ax = axes[1, 1]; ax.axis("off"); ax.set_title("GRPO vs PPO", fontsize=13, fontweight="bold")
t = ("PPO:                    GRPO:\n"
     "- 需要 Critic            - 不需要 Critic\n"
     "- 内存 2x               - 内存 1x\n"
     "- 通用                   - 可验证任务\n"
     "- InstructGPT            - DeepSeek-R1\n\n"
     "GRPO 核心优势:\n"
     "  省 Critic -> 节省50%显存\n"
     "  组内归一化天然基线\n"
     "  适合数学/代码任务")
ax.text(0.05, 0.95, t, transform=ax.transAxes, fontsize=8, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
plt.tight_layout(); plt.show()

In [ ]:
# 回复风格变化
print("训练过程中的回复风格变化")
stages = [
    ("Epoch 1", "15加8等于23", "23"),
    ("Epoch 3", "15 + 8 = 23，答案是 23", "23"),
    ("Epoch 5", "<think>15+8=23</think>\n答案: 23", "23"),
    ("Epoch 8", "<think>\n已知: 15个\n新增: 8个\n计算: 15+8=23\n验算: 23-8=15 ok\n</think>\n答案: 23", "23"),
]
for name, resp, ans in stages:
    t = combined_reward(resp, ans)
    print(f"\n--- {name} ---")
    print(f"回复: {resp}")
    print(f"综合奖励: {t:.2f}")

---
## 4. 涌现现象分析

GRPO 训练中模型学会了：
1. **结构化思考**：自发使用 think 标签
2. **多步推理**：分解问题
3. **自我验证**：验算
4. **清晰表达**：答案与推理分离

In [ ]:
# 涌现分析
print("训练前后对比")
test_qs = [("3个班每班25人共多少？", "75"), ("绳子50米剪18米剩多少？", "32"), ("96苹果分8人每人？", "12")]
pre = ["75人", "50减18等于32米", "每人12个"]
post = [
    "<think>\n班级:3 每班:25\n3x25=75\n</think>\n答案: 75",
    "<think>\n总长:50 剪去:18\n50-18=32\n验算:32+18=50 ok\n</think>\n答案: 32",
    "<think>\n总数:96 人数:8\n96/8=12\n验算:12x8=96 ok\n</think>\n答案: 12",
]
for i, ((q, a), pr, po) in enumerate(zip(test_qs, pre, post)):
    print(f"\n问题 {i+1}: {q}")
    print(f"  训练前: {pr} (奖励:{combined_reward(pr, a):.2f})")
    print(f"  训练后: {po[:60]}... (奖励:{combined_reward(po, a):.2f})")

In [ ]:
# think 标签使用率变化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 9)
think_usage = [0.02, 0.05, 0.15, 0.35, 0.55, 0.72, 0.85, 0.92]
accuracy = [0.15, 0.22, 0.30, 0.45, 0.58, 0.68, 0.75, 0.80]

ax = axes[0]
ax.plot(epochs, think_usage, "b-o", label="<think>使用率", linewidth=2)
ax.plot(epochs, accuracy, "r-s", label="答案正确率", linewidth=2)
ax.fill_between(epochs, think_usage, alpha=0.1, color="blue")
ax.set_title("<think> 使用率与正确率"); ax.set_xlabel("Epoch"); ax.set_ylabel("比率")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

ax = axes[1]
pre_r = [combined_reward(r, a) for r, a in zip(pre, [a for _, a in test_qs])]
post_r = [combined_reward(r, a) for r, a in zip(post, [a for _, a in test_qs])]
x = np.arange(3); w = 0.35
ax.bar(x-w/2, pre_r, w, label="训练前", color="#FF6B6B", alpha=0.8)
ax.bar(x+w/2, post_r, w, label="训练后", color="#4ECDC4", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([f"Q{i+1}" for i in range(3)])
ax.set_title("训练前后奖励对比"); ax.set_ylabel("综合奖励"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# 涌现分析总结
print("涌现分析：<think> 标签的自发出现")
print()
print("1. 初期: 不知道 <think>，直接输出答案，格式奖励~0")
print("2. 中期: 偶然出现类似标签 -> 高奖励 -> GRPO 增加概率")
print("3. 后期: 稳定使用 <think>，多步推理")
print()
print("这就是涌现：")
print("  我们没有教模型用 <think>")
print("  只告诉它 '用了格式好有奖励'")
print("  模型自己发现了最优格式策略")

In [ ]:
# 不同组大小 G 的影响
print("组大小 G 对训练的影响")
for G_val in [2, 4, 8, 16]:
    np.random.seed(42)
    q = 0.3; epoch_rewards = []
    for ep in range(8):
        rews = [0.3*np.random.beta(q*5+1,3)+0.7*np.random.beta(q*3+1,4) for _ in range(50*G_val)]
        epoch_rewards.append(np.mean(rews))
        q = min(q + 0.1, 0.95)
    print(f"  G={G_val:2d} | 最终奖励: {epoch_rewards[-1]:.4f}")

In [ ]:
# G 值对比可视化
fig, ax = plt.subplots(figsize=(10, 5))
for G_val, color in [(2, "red"), (4, "blue"), (8, "green"), (16, "purple")]:
    np.random.seed(42); q = 0.3; rg = []
    for ep in range(8):
        rg.append(np.mean([0.3*np.random.beta(q*5+1,3)+0.7*np.random.beta(q*3+1,4) for _ in range(50*G_val)]))
        q = min(q+0.1, 0.95)
    ax.plot(range(1,9), rg, "-o", color=color, label=f"G={G_val}", ms=5)
ax.set_title("不同组大小 G 的训练曲线"); ax.set_xlabel("Epoch"); ax.set_ylabel("奖励")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### 3.3 GRPOTrainer 关键配置

```python
GRPOConfig(
    num_generations=4,           # 每个prompt采样G个
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_completion_length=256,
    num_train_epochs=1,
    fp16=True,
)
```

**参数建议：**
- **num_generations**: 4~8，太小方差大，太大计算贵
- **learning_rate**: 1e-5~5e-6，比 SFT 小
- **max_completion_length**: 数学题 256 够用

In [ ]:
# 学习率影响
fig, ax = plt.subplots(figsize=(10, 5))
for lr_l, speed in [("1e-4 (太大)", 0.25), ("1e-5 (合适)", 0.12), ("1e-6 (太小)", 0.03)]:
    np.random.seed(42); q = 0.3; rews = []
    for ep in range(8):
        rews.append(np.mean([0.3*np.random.beta(q*5+1,3)+0.7*np.random.beta(q*3+1,4) for _ in range(200)]))
        q = min(q + speed, 0.95)
    ax.plot(range(1,9), rews, "-o", label=lr_l, ms=5)
ax.set_title("学习率对训练的影响"); ax.set_xlabel("Epoch"); ax.set_ylabel("奖励")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5. 总结

1. **DeepSeek-R1** 证明纯 RL 可以涌现推理能力
2. **GRPO** 组内归一化代替 Critic，节省 50% 显存
3. **Reward 设计**是关键：格式引导思考，正确性保证质量
4. **涌现现象**：模型自发学会结构化推理

### GRPO 训练配方
```
num_generations = 4~8      # 采样数G
learning_rate = 1e-5~5e-6
reward = 0.3 * format + 0.7 * correctness
```

### 思考题

1. **格式:正确性 = 0.3:0.7 的权重如何影响训练？** 设为 0.9:0.1 呢？

2. **为什么 DeepSeek-R1 选择不做 SFT，直接 GRPO？** 优缺点？

3. **组大小 G 如何影响训练？** 方差 vs 计算开销。

4. **GRPO 用于代码生成，reward 函数如何设计？**